## US STEEL: OPTION-IMPLIED DEAL PROBABILITIES

Nippon Steel agreed on 2023-12-18 to buy US Steel (X) for \$55 per share in cash. The
deal ran into political opposition, was blocked by President Biden on 2025-01-03, was
revived under President Trump, and closed on 2025-06-18. This notebook applies the
method in `report/report.typ` on key dates of that path: fit the break-state mixture to
X's puts, tied to the stock through the forward, and compare the implied completion
probability with the stock-only estimate.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# ----------------------------------------------------------------------------
# Imports
# ----------------------------------------------------------------------------

# Standard library imports
from datetime import UTC, date, datetime, time, timedelta
from pathlib import Path

# Third-party imports
import databento as db
import numpy as np
import plotly.graph_objects as go
import polars as pl
from scipy.optimize import minimize_scalar
from scipy.stats import norm

# First-party imports
from xpectral.data import DatabentoREST, USTreasuryRates

## Deal terms and key dates

The merger agreement's end date, as extended for regulatory approval, was 2025-06-18.
Each key date is a trading day on or right after a major event; the pair of September
2024 dates and the pair of May 2025 dates bracket a single piece of news.

In [3]:
ticker = "X"
W = 55.0  # total workout: cash per X share at closing
outside_date = date(2025, 6, 18)  # end date in the merger agreement, as extended

events = pl.DataFrame(
    {
        "date": [
            date(2023, 12, 18),
            date(2024, 3, 14),
            date(2024, 4, 12),
            date(2024, 9, 3),
            date(2024, 9, 4),
            date(2024, 12, 23),
            date(2025, 1, 3),
            date(2025, 4, 7),
            date(2025, 5, 22),
            date(2025, 5, 23),
            date(2025, 6, 13),
        ],
        "event": [
            "deal announced",
            "Biden: should remain American",
            "shareholders approve",
            "day before block report",
            "report: Biden to block",
            "CFIUS refers to Biden",
            "Biden blocks deal",
            "Trump orders new review",
            "day before partnership",
            "Trump announces partnership",
            "approval order signed",
        ],
    }
)
events

date,event
date,str
2023-12-18,"""deal announced"""
2024-03-14,"""Biden: should remain American"""
2024-04-12,"""shareholders approve"""
2024-09-03,"""day before block report"""
2024-09-04,"""report: Biden to block"""
…,…
2025-01-03,"""Biden blocks deal"""
2025-04-07,"""Trump orders new review"""
2025-05-22,"""day before partnership"""


## Data

One day of option quotes (consolidated best bid and offer, sampled each minute) per key
date, and NYSE daily bars for the stock. Massive's stock history does not reach back to
2023, so the bars also come from Databento. Both are cached in `data/`; the cost check is
free and the pulls only run for files that are missing.

In [4]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
databento = DatabentoREST()
starts = [datetime(d.year, d.month, d.day, tzinfo=UTC) for d in events["date"]]

cost = sum(
    databento.get_option_quotes_cost([ticker], s, s + timedelta(days=1)) for s in starts
)
print(f"option quotes, all key dates: ${cost:,.2f}")

for s in starts:
    path = data_dir / f"x-options-cbbo-1m-{s:%Y-%m-%d}.parquet"
    if not path.exists():
        quotes = databento.get_option_quotes([ticker], s, s + timedelta(days=1))
        quotes.collect().write_parquet(path)

bars_path = data_dir / "x-ohlcv-1d-xnys.parquet"
if not bars_path.exists():
    store = db.Historical().timeseries.get_range(
        dataset="XNYS.PILLAR",
        schema="ohlcv-1d",
        symbols=[ticker],
        start="2023-11-01",
        end="2025-06-20",
    )
    pl.from_pandas(store.to_df().reset_index()).write_parquet(bars_path)
bars = pl.read_parquet(bars_path).with_columns(date=pl.col("ts_event").dt.date())

S_pre = bars.filter(pl.col("date") < events["date"][0])["close"][-1]
print(f"last close before announcement: {S_pre:.2f}")

option quotes, all key dates: $0.59
last close before announcement: 39.33


In [6]:
bars

ts_event,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol,date
"datetime[ns, UTC]",u8,u16,u32,f64,f64,f64,f64,u64,str,date
2023-11-01 00:00:00 UTC,35,9,17590,33.86,34.29,33.52,34.25,1038124,"""X""",2023-11-01
2023-11-02 00:00:00 UTC,35,9,17590,34.19,34.36,33.95,34.24,779412,"""X""",2023-11-02
2023-11-03 00:00:00 UTC,35,9,17590,34.13,34.49,34.1,34.24,783736,"""X""",2023-11-03
2023-11-06 00:00:00 UTC,35,9,17590,34.3,34.55,34.205,34.51,881040,"""X""",2023-11-06
2023-11-07 00:00:00 UTC,35,9,17590,34.23,34.44,33.93,34.18,708812,"""X""",2023-11-07
…,…,…,…,…,…,…,…,…,…,…
2025-06-11 00:00:00 UTC,35,9,17590,53.8,54.0,53.4,53.48,1786021,"""X""",2025-06-11
2025-06-12 00:00:00 UTC,35,9,17590,53.45,54.115,53.41,53.71,1179151,"""X""",2025-06-12
2025-06-13 00:00:00 UTC,35,9,17590,51.95,52.7,51.47,52.19,2266088,"""X""",2025-06-13


Treasury par yields set the discount rate, interpolated to each option's maturity.
`USTreasuryRates` currently fails on ranges that start before 2024, so the December 2023
date uses the first January 2024 curve, two weeks later.

In [7]:
rates = USTreasuryRates().get_rates(date(2024, 1, 1), events["date"][-1]).collect()
tenor_cols = [
    "bc_1month",
    "bc_2month",
    "bc_3month",
    "bc_6month",
    "bc_1year",
    "bc_2year",
    "bc_5year",
    "bc_10year",
    "bc_30year",
]
tenor_years = np.array([1, 2, 3, 6, 12, 24, 60, 120, 360]) / 12


def rate(on: date, years: float) -> float:
    """Treasury par yield on the latest date up to ``on``, interpolated to ``years``."""
    curve = rates.filter(pl.col("date") <= max(on, rates["date"].min()))[-1]
    return float(np.interp(years, tenor_years, curve.select(tenor_cols).row(0))) / 100

## Model

For each key date we use the first expiry on or after the outside date, so the deal has
either closed or broken by expiry. If it closes, the options are cash-settled at $W$,
so a put struck below $W$ pays nothing. If it breaks, the stock is lognormal with mean
$\mu_\text{break}$ and log volatility $\sigma_\text{break}$. A put below $W$ is then
worth (report equation 3)

$$P(K) = \pi_\text{break}\, P_\text{B}(K;\ \mu_\text{break}, \sigma_\text{break}),$$

where $P_\text{B}$ is Black's put price. The forward ties the two states to the stock
(report equation 4):

$$S_0 e^{rT} - D = (1 - \pi_\text{break})\, W + \pi_\text{break}\, \mu_\text{break}.$$

Fitting the puts alone does not work: a higher break probability with a higher fallback
prices them almost the same as a lower probability with a lower fallback. The forward
removes that freedom by fixing $\mu_\text{break}$ for each $\pi_\text{break}$.

Fit choices:
- **Quotes:** each contract's last quote at or before the close. A missing bid counts as
  zero, so quotes with only an ask still bound the fit from above.
- **Weights:** errors are measured against the mid, in units of the bid–ask spread.
- **Strikes:** out-of-the-money puts only ($K \le S_0$). Listed options are American,
  and Black's European price would undervalue in-the-money puts.
- **Simplifications:** the close-state value is $W$ at expiry, ignoring interest earned
  between closing and expiry, and dividends are \$0.05 per quarter.
- **Uncertainty:** for every $\pi_\text{break}$ on a grid we fit the best
  $\sigma_\text{break}$. The band is the range of $\pi_\text{break}$ whose fit still
  prices the quotes within their bid–ask spread on average.

In [8]:
def black_put(F, K, sigma, discount):
    """Black put price for forward ``F``, strike ``K``, total volatility ``sigma``."""
    d1 = (np.log(F / K) + sigma**2 / 2) / sigma
    return discount * (K * norm.cdf(sigma - d1) - F * norm.cdf(-d1))


def closing_chain(quotes: pl.DataFrame) -> pl.DataFrame:
    """Each contract's last quote at or before 4 pm, with a missing bid set to zero."""
    return (
        quotes.filter(pl.col("ts_recv").dt.time() <= time(16, 0))
        .sort("ts_recv")
        .group_by("symbol")
        .last()
        .filter(pl.col("ask_px_00").is_not_null())
        .with_columns(bid=pl.col("bid_px_00").fill_null(0.0), ask=pl.col("ask_px_00"))
    )


def fit_snapshot(on: date) -> dict:
    """Fit the break-state mixture to out-of-the-money puts on ``on``."""
    S_0 = bars.filter(pl.col("date") == on)["close"][0]
    chain = closing_chain(pl.read_parquet(data_dir / f"x-options-cbbo-1m-{on}.parquet"))
    expiry = chain.filter(pl.col("expiration") >= outside_date)["expiration"].min()
    puts = chain.filter(
        (pl.col("expiration") == expiry)
        & (pl.col("option_type") == "P")
        & (pl.col("strike") <= S_0)
    ).sort("strike")

    T = (expiry - on).days / 365
    r = rate(on, T)
    discount = np.exp(-r * T)
    F = S_0 / discount - 0.05 * int(4 * T)
    K = puts["strike"].to_numpy()
    mid = ((puts["bid"] + puts["ask"]) / 2).to_numpy()
    spread = (puts["ask"] - puts["bid"]).to_numpy()

    def mean_loss(pi, sigma):
        mu = (F - (1 - pi) * W) / pi
        if mu <= 0:
            return np.inf
        model = pi * black_put(mu, K, sigma, discount)
        return np.mean(((model - mid) / spread) ** 2)

    # Profile over the break probability: best volatility for each value.
    grid = np.linspace(0.005, 0.995, 199)
    fits = [
        minimize_scalar(
            lambda s: mean_loss(pi, s), bounds=(0.05, 1.5), method="bounded"
        )
        for pi in grid
    ]
    losses = np.array([f.fun for f in fits])
    best = int(np.argmin(losses))
    pi = grid[best]
    # Within the spread on average: |model - mid| <= spread / 2, a mean loss of 0.25.
    inside = grid[losses <= 0.25]

    return {
        "date": on,
        "S_0": S_0,
        "expiry": expiry,
        "n_puts": len(K),
        "pi_break": pi,
        "pi_low": inside.min() if inside.size else None,
        "pi_high": inside.max() if inside.size else None,
        "mu_break": (F - (1 - pi) * W) / pi,
        "sigma_break": fits[best].x,
        "mean_loss": losses[best],
        "r": r,
        "T": T,
        "F": F,
    }

## Results

`p_options` is $1 - \pi_\text{break}$ with its band. `p_stock` is report equation (1)
with the pre-announcement close as the fallback, the usual stock-only estimate.

In [9]:
results = (
    pl.DataFrame([fit_snapshot(d) for d in events["date"]])
    .join(events, on="date")
    .with_columns(
        p_options=1 - pl.col("pi_break"),
        p_low=1 - pl.col("pi_high"),
        p_high=1 - pl.col("pi_low"),
        p_stock=(pl.col("S_0") - S_pre) / (W - S_pre),
    )
)
results.select(
    "date",
    "event",
    "S_0",
    "n_puts",
    "p_options",
    "p_low",
    "p_high",
    "mu_break",
    "sigma_break",
    "mean_loss",
    "p_stock",
).with_columns(pl.col(pl.Float64).round(3))

date,event,S_0,n_puts,p_options,p_low,p_high,mu_break,sigma_break,mean_loss,p_stock
date,str,f64,i64,f64,f64,f64,f64,f64,f64,f64
2023-12-18,"""deal announced""",49.59,15,0.855,0.005,0.94,39.956,0.498,0.044,0.655
2024-03-14,"""Biden: should remain American""",38.26,11,0.17,0.005,0.555,37.513,0.492,0.066,-0.068
2024-04-12,"""shareholders approve""",41.33,12,0.465,0.005,0.68,33.888,0.418,0.108,0.128
2024-09-03,"""day before block report""",35.6,10,0.295,0.005,0.495,29.122,0.524,0.082,-0.238
2024-09-04,"""report: Biden to block""",29.38,7,0.005,0.005,0.34,30.158,0.556,0.053,-0.635
…,…,…,…,…,…,…,…,…,…,…
2025-01-03,"""Biden blocks deal""",30.47,14,0.005,0.005,0.255,30.903,0.361,0.044,-0.565
2025-04-07,"""Trump orders new review""",44.5,28,0.415,0.005,0.635,37.722,0.356,0.118,0.33
2025-05-22,"""day before partnership""",42.9,34,0.29,0.005,0.435,38.168,0.236,0.104,0.228


In [10]:
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
labels = [f"{d:%Y-%m-%d}<br>{e}" for d, e in zip(results["date"], results["event"])]

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=labels,
        y=results["p_options"],
        mode="lines+markers",
        name="options (forward-tied mixture)",
        line={"width": 2, "color": colors[0]},
        error_y={
            "type": "data",
            "symmetric": False,
            "array": results["p_high"] - results["p_options"],
            "arrayminus": results["p_options"] - results["p_low"],
            "color": colors[0],
        },
    )
)
fig.add_trace(
    go.Scatter(
        x=labels,
        y=results["p_stock"],
        mode="lines+markers",
        name="stock only, pre-announcement fallback",
        line={"width": 2, "color": colors[1], "dash": "dot"},
    )
)
fig.add_hline(y=0, line={"width": 1, "color": "grey"})
fig.update_layout(
    title="Implied Probability That Nippon Steel Completes the US Steel Deal",
    yaxis_title="Completion probability",
    yaxis={"range": [-0.7, 1.05]},
    width=900,
    height=500,
    margin={"l": 60, "r": 20, "t": 50, "b": 140},
    legend={"x": 0.5, "xanchor": "center", "y": 1, "yanchor": "top"},
)
fig.write_image("output/x_completion_probability.png", scale=2)
fig.show()

## Fit and implied distribution around the partnership news

On 2025-05-22 and 2025-05-23, the day before and the day of Trump's partnership
announcement: the put quotes with the fitted prices, and the implied distribution of
X at expiry as a cumulative probability. It rises through the break hump, then jumps by
$1 - \pi_\text{break}$ at $W$, where the deal closes.

In [11]:
fig = go.Figure()
for color, on in zip(colors, [date(2025, 5, 22), date(2025, 5, 23)]):
    row = results.filter(pl.col("date") == on).row(0, named=True)
    chain = closing_chain(pl.read_parquet(data_dir / f"x-options-cbbo-1m-{on}.parquet"))
    puts = chain.filter(
        (pl.col("expiration") == row["expiry"])
        & (pl.col("option_type") == "P")
        & (pl.col("strike") <= row["S_0"])
    ).sort("strike")
    K = puts["strike"].to_numpy()
    fig.add_trace(
        go.Scatter(
            x=K,
            y=(puts["bid"] + puts["ask"]) / 2,
            mode="markers",
            name=f"{on} quotes (bid–ask)",
            marker={"color": color, "size": 5},
            error_y={
                "type": "data",
                "array": (puts["ask"] - puts["bid"]) / 2,
                "color": color,
                "thickness": 1,
            },
        )
    )
    model = row["pi_break"] * black_put(
        row["mu_break"], K, row["sigma_break"], np.exp(-row["r"] * row["T"])
    )
    fig.add_trace(
        go.Scatter(
            x=K,
            y=model,
            mode="lines",
            name=f"{on} fit",
            line={"width": 2, "color": color},
        )
    )
fig.update_layout(
    title="X Put Quotes and Fitted Prices, June 2025 Expiry",
    xaxis_title="Strike",
    yaxis_title="Put price",
    width=700,
    height=400,
    margin={"l": 60, "r": 20, "t": 50, "b": 50},
    legend={"x": 0, "xanchor": "left", "y": 1, "yanchor": "top"},
)
fig.write_image("output/x_put_fit.png", scale=2)
fig.show()

In [12]:
S = np.linspace(15.0, 60.0, 901)
fig = go.Figure()
for color, on in zip(colors, [date(2025, 5, 22), date(2025, 5, 23)]):
    row = results.filter(pl.col("date") == on).row(0, named=True)
    sigma = row["sigma_break"]
    m = np.log(row["mu_break"]) - sigma**2 / 2
    cdf = row["pi_break"] * norm.cdf((np.log(S) - m) / sigma) + (
        1 - row["pi_break"]
    ) * (S >= W)
    fig.add_trace(
        go.Scatter(
            x=S,
            y=cdf,
            mode="lines",
            name=f"{on}: break {row['pi_break']:.0%}, close {1 - row['pi_break']:.0%}",
            line={"width": 2, "color": color},
        )
    )
fig.add_vline(x=W, line={"width": 1, "dash": "dot", "color": "grey"})
fig.update_layout(
    title="Implied Probability That X Ends Below Each Price, June 2025 Expiry",
    xaxis_title="S_T",
    yaxis_title="Q(S_T ≤ S)",
    width=700,
    height=400,
    margin={"l": 60, "r": 20, "t": 50, "b": 50},
    legend={"x": 0, "xanchor": "left", "y": 1, "yanchor": "top"},
)
fig.write_image("output/x_implied_distribution.png", scale=2)
fig.show()

## Takeaways

- **The options track the news.** The implied completion probability is about 85% at
  announcement, falls to 17% after Biden's March 2024 statement, recovers to 46% at the
  shareholder vote, and drops from 30% to roughly zero on the September 2024 report
  that Biden would block the deal. It stays near zero through the block, rises to 42%
  when Trump orders a new review, and jumps from 29% to 80% on the partnership news.
- **The options fix the fallback.** Through late 2024 X traded below its
  pre-announcement close of \$39.33, partly because steel stocks fell. The stock-only
  formula then gives negative probabilities, which is the fallback problem from the
  report in its plainest form. The fit puts the fallback near \$30 in that period,
  close to where the stock traded, so the market was pricing X almost as a standalone
  company.
- **The two estimates agree when the naive fallback happens to be right.** In May and
  June 2025 the implied fallback is about \$41, near \$39.33, and both methods give
  about 80%.
- **The bands are wide.** On every date except 2025-05-23, the band runs from the
  best estimate down to zero completion probability. The quotes rule out high
  completion probabilities far better than low ones, so the upper end of each band
  is the more informative number. On 2025-06-13 no value fits within the spreads on
  average, so that date has no band.
- **These are risk-neutral probabilities.** The deal closed even though the options
  priced it near zero in January 2025. One outcome does not test calibration; that
  needs many deals, as the report's validation section describes.

Simplifications to revisit: the close-state value ignores interest between closing and
expiry, dividends are approximate, and only out-of-the-money puts enter the fit. Next
steps: fit every expiry for the failure term structure (timing), and add calls above $W$
for the topping-bid state.